# Clase 5: Optimización II — técnicas avanzadas y regularización

En esta clase retomas el descenso de gradiente que trabajaste en la clase anterior y resuelves su punto débil: la lentitud en paisajes de pérdida mal condicionados. Ejecuta cada celda de código a medida que avanzas por la explicación; este cuaderno está pensado para proyectarse y correr en vivo durante la sesión.

**Objetivos de aprendizaje**

Al terminar esta clase, sabes:

- Explicar por qué el descenso de gradiente puro oscila en valles estrechos
- Implementar la actualización con momento y comparar su trayectoria con la del descenso de gradiente puro
- Describir qué aporta la matriz Hessiana y por qué invertirla resulta costoso en redes con muchos parámetros
- Comparar SGD, SGD con momento y Adam sobre el mismo problema, en PyTorch
- Aplicar dropout y weight decay como técnicas de regularización en un modelo de PyTorch

## El problema de los valles estrechos

Recuerda la clase anterior: cuando el paisaje de pérdida tiene una dirección mucho más curvada que otra, el descenso de gradiente rebota entre las dos paredes del valle y avanza muy poco hacia la salida. Ese zigzag aparece porque el gradiente apunta con fuerza hacia las laderas y solo un poco hacia delante, que es la dirección en la que realmente quieres avanzar.

Dos síntomas del mismo problema:

- Si reduces la tasa de aprendizaje para no salirte del valle, avanzas de forma segura pero lenta
- Si la subes para avanzar más rápido, oscilas todavía más o diverges

Hoy resuelves este problema con dos ideas: el momento y, más adelante, la información de curvatura.

## Momento: una bola que acumula inercia

Imagina que sueltas una bola pesada en el mismo valle, en lugar de caminar mirando solo la pendiente bajo tus pies. La bola no reacciona a cada irregularidad del terreno: acumula la velocidad que ya llevaba, es decir, tiene inercia.

Las fuerzas que empujan la bola hacia una ladera y hacia la otra alternan de signo y tienden a cancelarse. La fuerza que la empuja a lo largo del cauce, hacia la salida, apunta siempre en la misma dirección y se refuerza paso a paso. El resultado: la bola rueda rápido a lo largo del valle y apenas rebota entre las paredes.

La regla de actualización con momento:

v_t = β · v_{t−1} + η · ∇L(θ_{t−1})

θ_t = θ_{t−1} − v_t

El coeficiente β (típicamente entre 0,9 y 0,99) controla cuánto "recuerda" la velocidad los gradientes anteriores. Compara ahora, a mano, la trayectoria de las dos versiones sobre un valle estrecho.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Función de pérdida con un valle estrecho: la dirección x está mucho
# más curvada que la dirección y, igual que en la analogía del valle
def perdida(x, y):
    return 25 * x**2 + y**2

def gradiente(x, y):
    return np.array([50 * x, 2 * y])

punto_inicial = np.array([-4.0, 8.0])
n_pasos = 60
tasa_aprendizaje = 0.018

# --- Descenso de gradiente puro ---
theta = punto_inicial.copy()
trayectoria_gd = [theta.copy()]
for _ in range(n_pasos):
    theta = theta - tasa_aprendizaje * gradiente(*theta)
    trayectoria_gd.append(theta.copy())
trayectoria_gd = np.array(trayectoria_gd)

# --- Descenso de gradiente con momento, implementado a mano ---
beta = 0.9
theta = punto_inicial.copy()
v = np.zeros(2)
trayectoria_momento = [theta.copy()]
for _ in range(n_pasos):
    v = beta * v + tasa_aprendizaje * gradiente(*theta)
    theta = theta - v
    trayectoria_momento.append(theta.copy())
trayectoria_momento = np.array(trayectoria_momento)

# --- Gráfico: curvas de nivel de la pérdida + las dos trayectorias ---
x_malla, y_malla = np.meshgrid(np.linspace(-5, 2, 200), np.linspace(-9, 9, 200))
z_malla = perdida(x_malla, y_malla)

plt.figure(figsize=(8, 6))
plt.contour(x_malla, y_malla, z_malla, levels=30, cmap="Greys", alpha=0.6)
plt.plot(trayectoria_gd[:, 0], trayectoria_gd[:, 1], "o-", markersize=3,
         label="Descenso de gradiente puro", color="tab:red")
plt.plot(trayectoria_momento[:, 0], trayectoria_momento[:, 1], "o-", markersize=3,
         label="Descenso de gradiente con momento", color="tab:blue")
plt.scatter([0], [0], color="black", marker="*", s=150, zorder=5, label="Mínimo")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Trayectoria en un valle estrecho: con y sin momento")
plt.legend()
plt.show()

print(f"Distancia al mínimo tras {n_pasos} pasos:")
print(f"  Sin momento: {np.linalg.norm(trayectoria_gd[-1]):.4f}")
print(f"  Con momento: {np.linalg.norm(trayectoria_momento[-1]):.4f}")

## Métodos de segundo orden y por qué son caros

El momento sigue usando solo el gradiente, la pendiente local. Los métodos de segundo orden usan además la curvatura del terreno, capturada por la matriz Hessiana, H, que reúne todas las derivadas parciales de segundo orden de la pérdida respecto a los parámetros:

H_ij = ∂²L(θ) / ∂θ_i ∂θ_j

Con esa información, el método de Newton calcula la actualización:

θ_{t+1} = θ_t − η · H⁻¹ · ∇L(θ_t)

En una dirección muy curvada, H⁻¹ reduce el paso; en una dirección poco curvada, lo amplía. En el valle estrecho, Newton "sabe" de antemano que una ladera es más pronunciada que la otra y ajusta el paso en una sola actualización, en lugar de necesitar cientos de pasos pequeños.

El problema es el coste. Para un modelo con n parámetros, H es una matriz de n × n:

- Almacenarla completa cuesta O(n²)
- Invertirla cuesta O(n³)

Una red modesta ya tiene varios millones de parámetros: elevar esa cifra al cubo da un número de operaciones inviable, incluso en el hardware más potente disponible hoy. Compáralo con el coste de una actualización de descenso de gradiente puro, que es O(n). Esa brecha explica por qué casi ninguna red profunda se entrena con Newton puro. Existen alternativas que reducen el coste renunciando a parte de la información de curvatura: los métodos cuasi-Newton, como BFGS (Broyden, Fletcher, Goldfarb y Shanno), y las aproximaciones diagonales por bloques o de Kronecker, conocidas como K-FAC (curvatura aproximada factorizada de Kronecker).

La siguiente celda mide, con NumPy, cuánto crece en la práctica el tiempo de invertir una matriz cuadrada al aumentar su tamaño.

In [ ]:
import time
import numpy as np

tamanos = [200, 400, 800, 1600]
tiempos = []

for n in tamanos:
    matriz = np.random.randn(n, n)
    matriz = matriz @ matriz.T + n * np.eye(n)  # la hacemos invertible y bien condicionada

    inicio = time.time()
    np.linalg.inv(matriz)
    tiempos.append(time.time() - inicio)

for n, t in zip(tamanos, tiempos):
    print(f"n = {n:>5}  ->  {t:.4f} segundos")

print()
print("Al duplicar n, el tiempo debería multiplicarse por, aproximadamente, ocho (2³),")
print("porque el coste de invertir una matriz crece como O(n³).")

## SGD, Momento y Adam cara a cara

Ahora compara los tres optimizadores sobre un problema con la misma curvatura mal condicionada de antes, pero con una red neuronal en lugar de un único punto. Entrenas tres veces el mismo perceptrón multicapa (MLP, por sus siglas en inglés), partiendo de los mismos pesos iniciales, y solo cambias el optimizador: el descenso de gradiente estocástico (SGD, por sus siglas en inglés) puro, SGD con momento (0,9) y Adam.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# --- Datos sintéticos con curvatura mal condicionada ---
# Una característica tiene mucha más varianza que la otra: esto genera
# un paisaje de pérdida en forma de valle estrecho, igual que en la analogía
n_muestras = 500
X = torch.randn(n_muestras, 2)
X[:, 0] *= 20.0

pesos_reales = torch.tensor([[3.0], [-2.0]])
y = X @ pesos_reales + 0.1 * torch.randn(n_muestras, 1)


def crear_mlp():
    """Perceptrón multicapa pequeño: dos capas ocultas de 16 neuronas.
    Fijamos la semilla para partir de los mismos pesos en las tres pruebas."""
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(2, 16),
        nn.ReLU(),
        nn.Linear(16, 16),
        nn.ReLU(),
        nn.Linear(16, 1),
    )


def entrenar(optimizador_fn, n_pasos=200):
    modelo = crear_mlp()
    perdida_fn = nn.MSELoss()  # error cuadrático medio (MSE, por sus siglas en inglés)
    optimizador = optimizador_fn(modelo.parameters())

    historial_perdida = []
    for _ in range(n_pasos):
        optimizador.zero_grad()
        pred = modelo(X)
        perdida = perdida_fn(pred, y)
        perdida.backward()
        optimizador.step()
        historial_perdida.append(perdida.item())

    return historial_perdida


tasa_aprendizaje = 0.001

resultados = {
    "SGD (sin momento)": entrenar(
        lambda params: torch.optim.SGD(params, lr=tasa_aprendizaje)
    ),
    "SGD con momento (0.9)": entrenar(
        lambda params: torch.optim.SGD(params, lr=tasa_aprendizaje, momentum=0.9)
    ),
    "Adam": entrenar(
        lambda params: torch.optim.Adam(params, lr=tasa_aprendizaje)
    ),
}

plt.figure(figsize=(8, 5))
for nombre, perdidas in resultados.items():
    plt.plot(perdidas, label=nombre)

plt.yscale("log")
plt.xlabel("Iteración")
plt.ylabel("Pérdida (MSE, escala log)")
plt.title("SGD, momento y Adam sobre el mismo MLP")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Regularización en la práctica

El optimizador resuelve la velocidad de convergencia; la regularización resuelve un problema distinto, el sobreajuste (overfitting), es decir, que el modelo memorice los datos de entrenamiento en lugar de generalizar a datos nuevos. Dos técnicas estándar:

- **Dropout.** En cada paso de entrenamiento, desactiva aleatoriamente un porcentaje de neuronas (habitualmente entre el 20 % y el 50 %). Esto impide que la red dependa en exceso de neuronas concretas. En PyTorch se añade con `torch.nn.Dropout`
- **Weight decay.** Añade una penalización proporcional a la magnitud de los pesos dentro de la función de pérdida, lo que empuja al modelo hacia soluciones más simples. En PyTorch se activa con el parámetro `weight_decay` de casi cualquier optimizador

**Combínalas** siempre que entrenes un modelo real: optimizador y regularización trabajan juntos, no de forma aislada.

In [ ]:
import torch.nn as nn
import torch.optim as optim

modelo_regularizado = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Dropout(0.3),      # desactiva el 30 % de las neuronas en cada paso de entrenamiento
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1),
)

optimizador_regularizado = optim.Adam(
    modelo_regularizado.parameters(),
    lr=0.001,
    weight_decay=1e-4,   # penaliza los pesos grandes dentro de la función de pérdida
)

# modelo_regularizado.train() activa el dropout; modelo_regularizado.eval() lo desactiva
modelo_regularizado.train()
salida_entrenamiento = modelo_regularizado(X)

modelo_regularizado.eval()
salida_evaluacion = modelo_regularizado(X)

print("Optimizador:", optimizador_regularizado)
print()
print("Salida en modo entrenamiento (primeras tres filas):")
print(salida_entrenamiento[:3])
print()
print("Salida en modo evaluación (primeras tres filas):")
print(salida_evaluacion[:3])

## Resumen de la clase

- El descenso de gradiente puro es lento u oscilante en paisajes de pérdida mal condicionados, como los valles estrechos
- El momento acumula una media móvil de los gradientes pasados: cancela las oscilaciones y refuerza el avance en direcciones consistentes
- La matriz Hessiana aporta información de curvatura, pero calcularla e invertirla cuesta O(n²) y O(n³) respectivamente, algo inviable en redes con millones de parámetros
- Existen aproximaciones (cuasi-Newton, diagonales por bloques, K-FAC) que reducen ese coste renunciando a parte de la información de curvatura
- Adam suele converger más rápido que SGD y que SGD con momento en problemas mal condicionados, porque además adapta la tasa de aprendizaje por parámetro
- Dropout y weight decay son técnicas de regularización que mejoran la generalización del modelo, no la velocidad de convergencia

**Abre ahora el notebook de práctica `../notebooks_practica/clase05_practica_optimizacion_avanzada.ipynb` y pon en práctica lo que acabas de ver.**